In [4]:
import requests
import json
import pandas as pd
from pathlib import Path
from IPython.display import display # 在 Notebook 中更美观地显示 DataFrame

# --- 1. 定义服务器和查询 ---

# 你的 FastAPI 服务器 URL
FASTAPI_SERVER_URL = "http://localhost:8000/graphql"

# 严格按照你提供的、已验证的查询结构
GRAPHQL_QUERY = """
query GetLatestLiquidations {
  liquidationCalls(first: 100, orderBy: timestamp, orderDirection: desc) {
    user {
      id
    }
    timestamp
    txHash
  }
}
"""

# 构建要发送的 JSON 负载
payload = {
    "query": GRAPHQL_QUERY
}

# --- 2. 定义输出路径 ---

# 你指定的精确输出目录
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data")

# 定义输出文件名 (保持不变)
OUTPUT_FILE = OUTPUT_DIR / "latest_100_liquidations.csv"


# --- 3. 执行查询、处理数据并保存 ---

print(f"正在向 {FASTAPI_SERVER_URL} 发送查询...")

try:
    # 确保目标目录存在
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 发送 HTTP POST 请求
    response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=20)

    # 检查 HTTP 错误
    response.raise_for_status()

    data = response.json()

    # 检查 The Graph 返回的业务逻辑错误
    if "errors" in data and data["errors"]:
        print("GraphQL 查询返回了错误：")
        print(json.dumps(data, indent=2))
    
    # 检查数据是否按预期格式返回
    elif "data" in data and "liquidationCalls" in data["data"]:
        records = data["data"]["liquidationCalls"]
        
        if not records:
            print("查询成功，但未返回任何清算数据。")
        else:
            print(f"成功获取 {len(records)} 条清算记录。")
            
            # 使用 json_normalize 来“展平”嵌套的 user.id
            df = pd.json_normalize(records)
            
            print("DataFrame 头部信息：")
            display(df.head())
            
            # --- 4. 保存数据 ---
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
            
            print(f"\nDataFrame 已成功保存到: {OUTPUT_FILE}")

    else:
        print("收到了未知的响应格式：")
        print(json.dumps(data, indent=2))

except requests.exceptions.ConnectionError:
    print(f"错误：无法连接到 {FASTAPI_SERVER_URL}")
    print("请确保你的 FastAPI 服务器 (uvicorn) F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data 正在运行！")

except requests.exceptions.RequestException as e:
    # 捕获所有其他的 requests 错误 (如超时, HTTP 错误等)
    print(f"请求过程中发生错误: {e}")

正在向 http://localhost:8000/graphql 发送查询...
成功获取 100 条清算记录。
DataFrame 头部信息：


,timestamp,txHash,user.id
0,1761799793,0xc4190d0663d44ec69fa1c2ef3657afc73d6c610cedf0...,0x86bb42aa9d25165a34c8dba94bddaf70e9516825
1,1761798533,0xa4871addc95f07db81844be1937fa76b6e01e32bf093...,0xc1ab8632e3f7ff2b62bcfc5c5deba3aaa21799c9
2,1761781215,0x40e9931a0d22af65521789770565ab3779c697e2cdfe...,0x084f247379c4106e2824686d3edb4a2fa837f38a
3,1761781067,0x9605667a6ce19c5f2952e098bc1c56badb66cb75bebc...,0x9892444271b9b238c88c7a3af651b8d10a74d95b
4,1761768703,0x3d98037c5625205cbdd66a8be6de5faeaceb614808ce...,0x18540d7f37179e01ed80e22375e2ccfb10191b7d



DataFrame 已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\latest_100_liquidations.csv


In [8]:
import requests
import json
import pandas as pd
from pathlib import Path
import time
from tqdm import tqdm  # <-- 这是唯一的修改！
from IPython.display import display

# --- 1. 定义路径和加载数据 ---

# 你的 FastAPI 服务器 URL
FASTAPI_SERVER_URL = "http://localhost:8000/graphql"

# 与上一个单元格相同的目录
DATA_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data")

# 我们在上一个单元格中创建的 CSV 文件
INPUT_FILE = DATA_DIR / "latest_100_liquidations.csv"
OUTPUT_FILE = DATA_DIR / "latest_100_liquidations_with_history.csv" 

print(f"正在从 {INPUT_FILE} 加载数据...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}")
    print("请确保上一个单元格已成功运行并生成了 CSV 文件。")
    # 如果文件不存在，停止执行
    raise

# 为了清晰起见，重命名列
if 'user.id' in df.columns:
    df.rename(columns={'user.id': 'user_id', 'timestamp': 'liquidation_timestamp'}, inplace=True)

print(f"成功加载 {len(df)} 条记录。")


# --- 2. 定义动态查询函数 ---

def build_user_history_query(user_id, before_timestamp):
    """
    根据用户ID和清算时间戳，构建 GraphQL 查询。
    """
    return f"""
    query GetUserLastAction {{
      user(id: "{user_id}") {{
        # 查询1：Reserves (存款/取款/借款/还款等)
        reserves(
          where: {{lastUpdateTimestamp_lt: {before_timestamp}}}
          first: 1
          orderBy: lastUpdateTimestamp
          orderDirection: desc
        ) {{
          lastUpdateTimestamp
        }}
        
        # 查询2：Emode 变更
        userEmodeSetHistory(
          first: 1
          orderBy: timestamp
          orderDirection: desc
          where: {{timestamp_lt: {before_timestamp}}}
        ) {{
          timestamp
        }}
        
        # 查询3：(更早的)清算
        liquidationCallHistory(
          first: 1
          orderBy: timestamp
          orderDirection: desc
          where: {{timestamp_lt: {before_timestamp}}}
        ) {{
          timestamp
        }}
      }}
    }}
    """

# --- 3. 遍历 DataFrame 并执行查询 ---

# 用于存储新数据的列表
last_action_timestamps = []
last_action_types = []

print(f"开始为 {len(df)} 个用户查询清算前历史记录...")

# 使用 tqdm 显示进度条
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="查询用户历史"):
    user_id = row['user_id']
    liq_ts = row['liquidation_timestamp']
    
    # 1. 构建查询和 payload
    query_string = build_user_history_query(user_id, liq_ts)
    payload = {"query": query_string}
    
    # 2. 发送请求
    try:
        response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=20)
        response.raise_for_status() # 检查 HTTP 错误
        data = response.json()
        
        # 3. 解析响应
        if "errors" in data and data["errors"]:
            print(f"用户 {user_id} 的 GraphQL 查询出错: {data['errors'][0]['message']}")
            last_action_timestamps.append(None)
            last_action_types.append(None)
        
        elif "data" in data and "user" in data["data"] and data["data"]["user"]:
            user_data = data["data"]["user"]
            
            # 存储所有可能的操作及其时间戳
            actions = {}
            
            # 注意：字段名与你的示例代码严格一致
            if user_data.get("reserves") and len(user_data["reserves"]) > 0:
                actions["reserves"] = user_data["reserves"][0]["lastUpdateTimestamp"]
                
            if user_data.get("userEmodeSetHistory") and len(user_data["userEmodeSetHistory"]) > 0:
                actions["userEmodeSetHistory"] = user_data["userEmodeSetHistory"][0]["timestamp"]
                
            if user_data.get("liquidationCallHistory") and len(user_data["liquidationCallHistory"]) > 0:
                actions["liquidationCallHistory"] = user_data["liquidationCallHistory"][0]["timestamp"]

            # 4. 比较并找出最后的操作
            if not actions:
                # 如果所有列表都为空，说明在清算前没有找到任何历史记录
                last_action_timestamps.append(None)
                last_action_types.append("No History Found")
            else:
                # 找到时间戳最大的操作类型
                last_action_type = max(actions, key=actions.get)
                last_action_timestamp = actions[last_action_type]
                
                last_action_timestamps.append(last_action_timestamp)
                last_action_types.append(last_action_type)
        
        else:
            # 成功，但没有返回 user 数据 (可能是子图没有该用户?)
            last_action_timestamps.append(None)
            last_action_types.append("User Not Found in Subgraph")
            
    except requests.exceptions.RequestException as e:
        print(f"用户 {user_id} 的请求失败: {e}")
        last_action_timestamps.append(None)
        last_action_types.append("Request Error")
    
    # 礼貌性暂停，避免请求过于频繁
    time.sleep(0.1)

print("所有查询已完成。")

# --- 4. 更新 DataFrame 并保存 ---

df['last_action_timestamp'] = last_action_timestamps
df['last_action_type'] = last_action_types

# 为了在 CSV 中更易读，将 Unix 时间戳转换为日期时间字符串
df['liquidation_datetime'] = pd.to_datetime(df['liquidation_timestamp'], unit='s', errors='coerce')
df['last_action_datetime'] = pd.to_datetime(df['last_action_timestamp'], unit='s', errors='coerce')


print("\nDataFrame 更新完毕，头部信息如下：")
display(df.head())

# 保存到 *新* 文件（这比覆盖更安全）
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n数据已成功保存到: {OUTPUT_FILE}")

正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\latest_100_liquidations.csv 加载数据...
成功加载 100 条记录。
开始为 100 个用户查询清算前历史记录...


查询用户历史: 100%|██████████| 100/100 [04:05<00:00,  2.45s/it]

所有查询已完成。

DataFrame 更新完毕，头部信息如下：


,liquidation_timestamp,txHash,user_id,last_action_timestamp,last_action_type,liquidation_datetime,last_action_datetime
0,1761799793,0xc4190d0663d44ec69fa1c2ef3657afc73d6c610cedf0...,0x86bb42aa9d25165a34c8dba94bddaf70e9516825,NaN,No History Found,2025-10-30 04:49:53,NaT
1,1761798533,0xa4871addc95f07db81844be1937fa76b6e01e32bf093...,0xc1ab8632e3f7ff2b62bcfc5c5deba3aaa21799c9,1.761019e+09,reserves,2025-10-30 04:28:53,2025-10-21 04:02:49
2,1761781215,0x40e9931a0d22af65521789770565ab3779c697e2cdfe...,0x084f247379c4106e2824686d3edb4a2fa837f38a,1.761595e+09,reserves,2025-10-29 23:40:15,2025-10-27 19:55:39
3,1761781067,0x9605667a6ce19c5f2952e098bc1c56badb66cb75bebc...,0x9892444271b9b238c88c7a3af651b8d10a74d95b,1.759443e+09,reserves,2025-10-29 23:37:47,2025-10-02 22:08:53
4,1761768703,0x3d98037c5625205cbdd66a8be6de5faeaceb614808ce...,0x18540d7f37179e01ed80e22375e2ccfb10191b7d,1.761335e+09,reserves,2025-10-29 20:11:43,2025-10-24 19:50:17



数据已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\latest_100_liquidations_with_history.csv


In [10]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# --- 1. 定义路径 ---

# 你的数据目录
DATA_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data")

# 我们在上一个单元格中创建的输入文件
INPUT_FILE = DATA_DIR / "latest_100_liquidations_with_history.csv"

# 你指定的新输出文件名
OUTPUT_FILE = DATA_DIR / "latest_100_liquidations_with_history_cleaned.csv"

# --- 2. 加载和筛选数据 ---

print(f"正在从 {INPUT_FILE} 加载数据...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}")
    print("请确保上一个单元格已成功运行并生成了 CSV 文件。")
    # 如果文件不存在，停止执行
    raise

print(f"原始数据集有 {len(df)} 行。")

# 筛选条件：保留 "last_action_type" 不等于 "No History Found" 的所有行
# 我们也顺便排除掉 "User Not Found in Subgraph" 和 "Request Error"（如果它们存在的话）
original_row_count = len(df)
df_cleaned = df[~df['last_action_type'].isin(["No History Found", "User Not Found in Subgraph", "Request Error"])].copy()

# .copy() 是一个好习惯，可以避免 Pandas 的 SettingWithCopyWarning

# --- 3. 显示结果并保存 ---

rows_removed = original_row_count - len(df_cleaned)
print(f"已移除 {rows_removed} 行 (类型为 No History Found, User Not Found 或 Error)。")
print(f"清洗后的数据集剩余 {len(df_cleaned)} 行。")

print("\n清洗后数据的头部信息：")
display(df_cleaned.head())

# 保存清洗后的 DataFrame
df_cleaned.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n清洗后的数据已成功保存到: {OUTPUT_FILE}")

正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\latest_100_liquidations_with_history.csv 加载数据...
原始数据集有 100 行。
已移除 40 行 (类型为 No History Found, User Not Found 或 Error)。
清洗后的数据集剩余 60 行。

清洗后数据的头部信息：


,liquidation_timestamp,txHash,user_id,last_action_timestamp,last_action_type,liquidation_datetime,last_action_datetime
1,1761798533,0xa4871addc95f07db81844be1937fa76b6e01e32bf093...,0xc1ab8632e3f7ff2b62bcfc5c5deba3aaa21799c9,1.761019e+09,reserves,2025-10-30 04:28:53,2025-10-21 04:02:49
2,1761781215,0x40e9931a0d22af65521789770565ab3779c697e2cdfe...,0x084f247379c4106e2824686d3edb4a2fa837f38a,1.761595e+09,reserves,2025-10-29 23:40:15,2025-10-27 19:55:39
3,1761781067,0x9605667a6ce19c5f2952e098bc1c56badb66cb75bebc...,0x9892444271b9b238c88c7a3af651b8d10a74d95b,1.759443e+09,reserves,2025-10-29 23:37:47,2025-10-02 22:08:53
4,1761768703,0x3d98037c5625205cbdd66a8be6de5faeaceb614808ce...,0x18540d7f37179e01ed80e22375e2ccfb10191b7d,1.761335e+09,reserves,2025-10-29 20:11:43,2025-10-24 19:50:17
5,1761763267,0x80903a02966c2a0974f9ae4641db6aff481c5d0b0945...,0xd8d6c693603729fdfed05a8243777ac6cb213508,1.761708e+09,reserves,2025-10-29 18:41:07,2025-10-29 03:12:51



清洗后的数据已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\latest_100_liquidations_with_history_cleaned.csv


### 还是要加txhash的

这是一个整合后的单元格代码，它会按顺序执行所有步骤：

第 1 步：查询 100 次清算，确保获取 user.id, timestamp, 和 txHash。

第 2 步：将结果（包含 txHash）加载到 DataFrame 中，然后循环遍历，查询每个用户的最后操作。

第 3 步：将历史记录列添加到 DataFrame 中（此时 txHash 仍然保留）。

第 4 步：过滤掉 "No History Found" 的行。

第 5 步：将这个最终的、干净的 DataFrame 保存为 Target_sample.csv。

In [12]:
import requests
import json
import pandas as pd
from pathlib import Path
import time
from tqdm import tqdm
from IPython.display import display
import sys

# --- 1. 定义常量和路径 ---

# 你的 FastAPI 服务器 URL
FASTAPI_SERVER_URL = "http://localhost:8000/graphql"

# 你指定的精确输出目录
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data")

# 最终的输出文件名
FINAL_OUTPUT_FILE = OUTPUT_DIR / "Target_sample.csv"

# --- 2. (第一步): 获取 100 条最新清算记录 ---

print("--- 第 1 部分：正在获取 100 条最新清算记录 ---")

# 严格按照你提供的、已验证的查询结构
# 确保 txHash 包含在内
GRAPHQL_QUERY = """
query GetLatestLiquidations {
  liquidationCalls(first: 100, orderBy: timestamp, orderDirection: desc) {
    user {
      id
    }
    timestamp
    txHash
  }
}
"""

payload = {
    "query": GRAPHQL_QUERY
}

try:
    # 确保目标目录存在
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 发送 HTTP POST 请求
    response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=20)
    response.raise_for_status() # 检查 HTTP 错误
    data = response.json()

    # 检查 GraphQL 错误
    if "errors" in data and data["errors"]:
        print("GraphQL 查询返回了错误：")
        print(json.dumps(data, indent=2))
        # 如果第一步失败，则停止执行
        raise ValueError("GraphQL query failed, stopping execution.")
    
    # 检查数据是否按预期格式返回
    elif "data" in data and "liquidationCalls" in data["data"]:
        records = data["data"]["liquidationCalls"]
        
        if not records:
            print("查询成功，但未返回任何清算数据。停止执行。")
            # 如果没有数据，也停止
            raise ValueError("No liquidation data found, stopping execution.")
        else:
            print(f"成功获取 {len(records)} 条清算记录。")
            
            # 使用 json_normalize 来“展平”嵌套的 user.id
            df = pd.json_normalize(records)
            
            # 重命名列以便后续使用
            df.rename(columns={'user.id': 'user_id', 'timestamp': 'liquidation_timestamp'}, inplace=True)
            
            print("DataFrame 头部信息：")
            display(df.head())

    else:
        print("收到了未知的响应格式：")
        print(json.dumps(data, indent=2))
        raise ValueError("Unknown response format, stopping execution.")

except requests.exceptions.ConnectionError:
    print(f"错误：无法连接到 {FASTAPI_SERVER_URL}")
    print("请确保你的 FastAPI 服务器 (uvicorn) 正在运行！")
    # 停止单元格
    raise
except requests.exceptions.RequestException as e:
    print(f"请求过程中发生错误: {e}")
    # 停止单元格
    raise


# --- 3. (第二步): 定义历史查询函数并遍历 DataFrame ---

print("\n--- 第 2 部分：正在查询每个用户的清算前历史记录 ---")

def build_user_history_query(user_id, before_timestamp):
    """
    根据用户ID和清算时间戳，构建 GraphQL 查询。
    """
    return f"""
    query GetUserLastAction {{
      user(id: "{user_id}") {{
        reserves(
          where: {{lastUpdateTimestamp_lt: {before_timestamp}}}
          first: 1
          orderBy: lastUpdateTimestamp
          orderDirection: desc
        ) {{
          lastUpdateTimestamp
        }}
        userEmodeSetHistory(
          first: 1
          orderBy: timestamp
          orderDirection: desc
          where: {{timestamp_lt: {before_timestamp}}}
        ) {{
          timestamp
        }}
        liquidationCallHistory(
          first: 1
          orderBy: timestamp
          orderDirection: desc
          where: {{timestamp_lt: {before_timestamp}}}
        ) {{
          timestamp
        }}
      }}
    }}
    """

# 用于存储新数据的列表
last_action_timestamps = []
last_action_types = []

# 使用 tqdm 显示进度条
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="查询用户历史"):
    user_id = row['user_id']
    liq_ts = row['liquidation_timestamp']
    
    query_string = build_user_history_query(user_id, liq_ts)
    payload = {"query": query_string}
    
    try:
        response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=20)
        response.raise_for_status()
        data = response.json()
        
        if "errors" in data and data["errors"]:
            last_action_timestamps.append(None)
            last_action_types.append("GraphQL Error")
        
        elif "data" in data and "user" in data["data"] and data["data"]["user"]:
            user_data = data["data"]["user"]
            actions = {}
            
            if user_data.get("reserves") and len(user_data["reserves"]) > 0:
                actions["reserves"] = user_data["reserves"][0]["lastUpdateTimestamp"]
            if user_data.get("userEmodeSetHistory") and len(user_data["userEmodeSetHistory"]) > 0:
                actions["userEmodeSetHistory"] = user_data["userEmodeSetHistory"][0]["timestamp"]
            if user_data.get("liquidationCallHistory") and len(user_data["liquidationCallHistory"]) > 0:
                actions["liquidationCallHistory"] = user_data["liquidationCallHistory"][0]["timestamp"]

            if not actions:
                last_action_timestamps.append(None)
                last_action_types.append("No History Found")
            else:
                last_action_type = max(actions, key=actions.get)
                last_action_timestamp = actions[last_action_type]
                last_action_timestamps.append(last_action_timestamp)
                last_action_types.append(last_action_type)
        else:
            last_action_timestamps.append(None)
            last_action_types.append("User Not Found in Subgraph")
            
    except requests.exceptions.RequestException as e:
        last_action_timestamps.append(None)
        last_action_types.append("Request Error")
    
    time.sleep(0.1)

print("所有历史查询已完成。")

# 将列表添加为 DataFrame 的新列
df['last_action_timestamp'] = last_action_timestamps
df['last_action_type'] = last_action_types


# --- 4. (第三步): 清洗数据 ---

print("\n--- 第 3 部分：正在清洗数据 ---")

original_row_count = len(df)
# 筛选条件：保留不等于 "No History Found"、"GraphQL Error"等的行
df_cleaned = df[~df['last_action_type'].isin(["No History Found", "User Not Found in Subgraph", "Request Error", "GraphQL Error"])].copy()

rows_removed = original_row_count - len(df_cleaned)
print(f"已移除 {rows_removed} 行 (类型为 No History Found, User Not Found 或 Error)。")
print(f"清洗后的数据集剩余 {len(df_cleaned)} 行。")


# --- 5. (第四步): 格式化并保存最终文件 ---

print("\n--- 第 4 部分：正在保存最终文件 ---")

# 添加易读的日期时间列
df_cleaned['liquidation_datetime'] = pd.to_datetime(df_cleaned['liquidation_timestamp'], unit='s', errors='coerce')
df_cleaned['last_action_datetime'] = pd.to_datetime(df_cleaned['last_action_timestamp'], unit='s', errors='coerce')

print("\n最终清洗后数据的头部信息：")
display(df_cleaned.head())

# 保存最终的 DataFrame
df_cleaned.to_csv(FINAL_OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n✅ 最终数据已成功保存到: {FINAL_OUTPUT_FILE}")

--- 第 1 部分：正在获取 100 条最新清算记录 ---
成功获取 100 条清算记录。
DataFrame 头部信息：


,liquidation_timestamp,txHash,user_id
0,1761831755,0xa8e5f15f90e36332cda79e7087e45de1f487247ddb4e...,0xd8d6c693603729fdfed05a8243777ac6cb213508
1,1761831755,0xbcc859f590e76955311cef493195fa41be1731e1dfc7...,0xd86bb2b25493e58d17c3f3e59b5e97e1c9da09e8
2,1761831433,0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f0...,0x64d920358366a309c6d0363b361a18a7f81855ff
3,1761828723,0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc...,0x33cf8f585e7063e31ec34f85721a65f4659f7172
4,1761828723,0xbb13e78bd55c4f8e30bb2026e6e9597af326c6a60ae3...,0x80a7dd43bf57aa72214578dc76857cf369243340



--- 第 2 部分：正在查询每个用户的清算前历史记录 ---


查询用户历史: 100%|██████████| 100/100 [04:06<00:00,  2.47s/it]

所有历史查询已完成。

--- 第 3 部分：正在清洗数据 ---
已移除 39 行 (类型为 No History Found, User Not Found 或 Error)。
清洗后的数据集剩余 61 行。

--- 第 4 部分：正在保存最终文件 ---

最终清洗后数据的头部信息：


,liquidation_timestamp,txHash,user_id,last_action_timestamp,last_action_type,liquidation_datetime,last_action_datetime
0,1761831755,0xa8e5f15f90e36332cda79e7087e45de1f487247ddb4e...,0xd8d6c693603729fdfed05a8243777ac6cb213508,1.761763e+09,liquidationCallHistory,2025-10-30 13:42:35,2025-10-29 18:41:07
2,1761831433,0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f0...,0x64d920358366a309c6d0363b361a18a7f81855ff,1.761584e+09,reserves,2025-10-30 13:37:13,2025-10-27 16:48:57
3,1761828723,0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc...,0x33cf8f585e7063e31ec34f85721a65f4659f7172,1.761067e+09,reserves,2025-10-30 12:52:03,2025-10-21 17:19:03
4,1761828723,0xbb13e78bd55c4f8e30bb2026e6e9597af326c6a60ae3...,0x80a7dd43bf57aa72214578dc76857cf369243340,1.761754e+09,liquidationCallHistory,2025-10-30 12:52:03,2025-10-29 16:06:01
5,1761828573,0xc45a34f681ae8c1ec34edd9baebf705f941772f32775...,0xffc409d0074d411074ff282cb93ed5a2ffcafd76,1.760691e+09,liquidationCallHistory,2025-10-30 12:49:33,2025-10-17 08:53:55



✅ 最终数据已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\Target_sample.csv
